In [ ]:
import emcee
import time
import numpy as np
import sys
import pandas as pd
import os
import subprocess as sp
import struct

if '../../SelfCalGroupFinder/py/' not in sys.path:
    sys.path.append('../../SelfCalGroupFinder/py/')
from calibrationdata import CalibrationData
from dataloc import PARAMS_BGSY1_FOLDER
from footprintmanager import FootprintManager
from pyutils import DEGREES_ON_SPHERE, get_volume_at_z

LATSHAM_BIN = "/mount/sirocco1/imw2293/GROUP_CAT/latsham/build/latsham"

### For peeking at halo properties

In [ ]:
HALO_CATALOG = "/mount/sirocco1/imw2293/GROUP_CAT/DATA/POPMOCK/smdpl_z0.19717.M8E9.C.h5";
# contains ['halos'], find it's columns and read it all in

import h5py
with h5py.File(HALO_CATALOG, 'r') as f:
    halos = f['halos']['table']
    print(halos.dtype.names)
    halos_data = halos[:]


print(len(halos_data))

In [ ]:
halos_data

### MCMC

In [ ]:
frac_area = FootprintManager().get_footprint("Y1", min_passes=3) / DEGREES_ON_SPHERE
caldata =  CalibrationData(PARAMS_BGSY1_FOLDER, np.array([-20, -21]), np.array([True]), 19.5, frac_area)
mag_starts = [17, 18, 19, 20, 21, 22]
colors = ['red', 'blue']

def calc_wp_from_mocks():
    t1 = time.time()
    
    # Define the path to the Corrfunc binary
    corrfunc_path = "/export/sirocco1/tinker/src/Corrfunc/bin/"
    if not os.path.exists(os.path.join(corrfunc_path, "wp")):
        corrfunc_path = "/mount/sirocco1/tinker/src/Corrfunc/bin/"


    rpbins_file = "/mount/sirocco1/imw2293/GROUP_CAT/SelfCalGroupFinder/py/parameters/bgs_y1/wp_rbins.dat"
    output_folder = "/mount/sirocco1/imw2293/GROUP_CAT/OUTPUT/LATSHAM/"

    nthreads = os.cpu_count()
    pimax = 40
    boxsize = 400 # Mpc/h

    # Force a sync to ensure the mock files from the C++ process are fully written to disk
    # before we try to read them with corrfunc. This prevents a potential race condition.
    sp.run(["sync"], check=True)

    if not os.path.exists(f"{corrfunc_path}/wp"):
        print(f"Corrfunc wp executable not found at {corrfunc_path}/wp. Skipping wp calculation.")
        return

    # Call corrfunc compiled executable to compute wp on the mock that was populated with the HOD extracted from this catalog
    for i in range(len(mag_starts)):
        for c in colors:
            m = abs(mag_starts[i])

            cmd = f"{corrfunc_path}/wp {boxsize} mock_{c}_M{m}.dat a {rpbins_file} {pimax} {nthreads} > wp_mock_{c}_M{m}.dat 2> wp_stderr.txt"
            result = sp.run(cmd, cwd=output_folder, shell=True, check=True)
            if result.returncode != 0:
                print(f"Error running corrfunc!")

    t2 = time.time()

    print(f"Calculated all wp in {t2 - t1:.2f} seconds.")

def chisqr():
    dof = 0
    chivec = []

    for mag in mag_starts:
        for c in colors:
                    
            # Read in the wp data from the mock and the data, and calculate chi^2
            output_mock = np.loadtxt("/mount/sirocco1/imw2293/GROUP_CAT/OUTPUT/LATSHAM/wp_mock_%s_M%d.dat" % (c, mag))
            output_data = np.loadtxt("/mount/sirocco1/imw2293/GROUP_CAT/SelfCalGroupFinder/py/parameters/bgs_y1/wp_%s_M%d.dat" % (c, mag))

            r = output_data[:,0]
            wp = output_data[:,1]
            wp_err = output_data[:,2]

            wp_m = output_mock[:,4]

            # If anything in nan, immediatly make these params very bad chisqr
            if np.isnan(wp).any() or np.isnan(wp_err).any() or np.isnan(wp_m).any():
                return np.inf

            # Because no sats, let's skip the 1 halo term area. Drop first 6 bins
            r = r[6:]
            wp = wp[6:]
            wp_err = wp_err[6:]
            wp_m = wp_m[6:]

            fsky = FootprintManager().get_footprint("Y1", min_passes=1) / DEGREES_ON_SPHERE # what went into clusting measurements
            volume = get_volume_at_z(caldata.zmaxes[0], fsky)
            vfac = (volume / 400.0**3)**.5 # factor by which to multiply errors
            efac = 0.1 # TODO tweak eventually
            wp_m_err = vfac*wp_err + efac*wp_m

            sigma = wp_err**2 + wp_m_err**2
            #print(f"Data err: {wp_err}. Mock err: {wp_m_err}")

            chivec.append(np.sum(((wp - wp_m) / sigma)**2))

    chi2 = np.sum(chivec)
    return chi2

In [ ]:
# Must keep this protocol syncronized with the C++ code in groups.hpp
MSG_REQUEST = 0
MSG_COMPLETED = 6
MSG_ABORTED = 7
TYPE_FLOAT = 0
TYPE_DOUBLE = 1

def lnlike(params):
    
    # Send parameters to latsham
    msg = struct.pack("<BBI", MSG_REQUEST, TYPE_DOUBLE, len(params)) + struct.pack(f"<{len(params)}d", *params)
    proc.stdin.write(msg)
    proc.stdin.flush()

    while proc.poll() is None: 
        header = pipe_reader.read(6)
        if len(header) == 0:
            continue
        if len(header) < 6:
            raise Exception("Incomplete header, len=" + str(len(header)))
        
        msg_type, data_type, count = struct.unpack("<BBI", header)

        if msg_type not in (MSG_FSAT, MSG_LHMR, MSG_LSAT, MSG_HOD, MSG_HODFIT, MSG_COMPLETED, MSG_ABORTED):
            raise Exception("Unexpected response")
        if data_type not in (TYPE_FLOAT, TYPE_DOUBLE):
            raise Exception("Unexpected data type")
        dtype_marker = 'd' if data_type == TYPE_DOUBLE else 'f'
        dtype = np.dtype(dtype_marker)
        bytes_needed = count * (8 if data_type == TYPE_DOUBLE else 4)

        payload = pipe_reader.read(bytes_needed)
        if len(payload) < bytes_needed:
            raise Exception(f"Incomplete payload, expected {bytes_needed} bytes but got {len(payload)} bytes") 
        data = struct.unpack(f"<{count}{dtype_marker}", payload)
        assert count == len(data)

        if msg_type == MSG_COMPLETED:
            #print("Latent SHAM completed successfully.")
            calc_wp_from_mocks()
            return -0.5 * chisqr()
            
        elif msg_type == MSG_ABORTED:
            raise Exception(f"Group Finder was aborted.")
        
        else:
            raise Exception(f"Unexpected message type: {msg_type}")

    raise Exception(f"Group Finder process ended. What happened?")


def lnprior(params):
    if len(params) == 1:
        scatter = params[0]
        if scatter < 0.00001 or scatter > 10.0:
            return -np.inf
    if len(params) == 4:
        for p in params:
            if p < -10.0 or p > 10.0:
                return -np.inf
    return 0.0

def lnprob(params):
    lp = lnprior(params)
    if lp == -np.inf:
        return -np.inf
    return lp + lnlike(params)

In [ ]:
read_fd, write_fd = os.pipe()
proc = sp.Popen([LATSHAM_BIN, "-P", str(write_fd)], stdin=sp.PIPE, pass_fds=(write_fd,))
os.close(write_fd)
pipe_reader = os.fdopen(read_fd, "rb")

In [ ]:
backfile = "latsham_mcmc_L2p2p_test.h5"
backend = emcee.backends.HDFBackend(backfile)
nwalkers = 10
ndim = 4
sampler = emcee.EnsembleSampler(nwalkers, ndim, lnprob, backend=backend)
init_pos = 1.0 * np.random.randn(nwalkers, ndim)

In [ ]:
# RUN MCMC
t0 = time.time()
sampler.run_mcmc(init_pos, 10, progress=True)

chain = sampler.get_chain()
t1 = time.time()

print(f"Generated {len(chain)} samples in {t1 - t0:.2f} seconds.")

In [ ]:
# Kill latsham if needed
if proc.poll() is None:
    print("Process is still running")
    # Kill it
    proc.terminate()
else:
    print(f"Process finished with code {proc.returncode}")
    # Note -15 is SIGTERM which is what happens when you kill, which is terminate() above

In [ ]:
chains = sampler.get_chain().reshape(-1, ndim)
logprobs = sampler.get_log_prob().reshape(-1)

In [ ]:
import corner
fig = corner.corner(chains.reshape(-1, ndim))

In [83]:
-0.5*logprobs.reshape(-1, nwalkers)

array([[  -0.        ,   -0.        ,   -0.        ,   -0.        ,
          -0.        ,   -0.        ,   -0.        ,   -0.        ,
          -0.        ,   -0.        ],
       [  -0.        ,   -0.        ,   -0.        ,   -0.        ,
          -0.        ,   -0.        ,   -0.        ,   -0.        ,
          -0.        ,   -0.        ],
       [  -0.        ,   -0.        ,   -0.        ,   -0.        ,
          -0.        ,   -0.        ,   -0.        ,   -0.        ,
          -0.        ,   -0.        ],
       [  -0.        ,   -0.        ,   -0.        ,   -0.        ,
          -0.        ,   -0.        ,   -0.        ,   -0.        ,
          -0.        ,   -0.        ],
       [  -0.        ,   -0.        ,   -0.        ,   -0.        ,
          -0.        ,   -0.        ,   -0.        ,   -0.        ,
          -0.        ,   -0.        ],
       [  -0.        ,   -0.        ,   -0.        ,   -0.        ,
          -0.        ,   -0.        ,   -0.        ,   -0

In [79]:
# Show best parameter and chisqr
best_idx = np.argmax(logprobs)
best_params = chains.reshape(-1, ndim)[best_idx]
print(f"Best fit params: {best_params}")
print(f"Best fit chi^2: {-2*logprobs[best_idx]}")


Best fit params: [ 0.31041663  0.09692983  0.21029271 -1.16355457]
Best fit chi^2: -0.0


In [80]:
# Worst model
worst_idx = np.argmin(logprobs)
worst_params = chains.reshape(-1, ndim)[worst_idx]
print(f"Worst fit params: {worst_params}")
print(f"Worst fit chi^2: {-2*logprobs[worst_idx]}")

Worst fit params: [ 0.50212246  0.71233582  0.32934982 -1.29609924]
Worst fit chi^2: inf


In [ ]:
# TODO why is chisqr -0.0 sometimes
